# Week 8: Web Scraping (part 2)
Harley Ungar
Oct 25, 2026

In [14]:
import csv
import json
import os # operating system library, let's us do things like making new folders
import requests
from time import sleep # gives us access to the sleep function that lets us deal with crawl delay
from bs4 import BeautifulSoup # the scraping library!


SCRAPED_PAGES_DIRECTORY = "saved_pages/" # new folder: files are now named by URL, so each page always matches its URL
SCRAPED_PAGES_PREFIX = "page"
scraped_pages_count = 0 # keep track of how many pages we have scraped

headers = {"User-Agent": "INFO664-class-project (educational)"}   # you can call this whatever you want, it shows up on the scraped website's logs as who you are (ie not a Google/AI bot or web browser)

os.makedirs("saved_pages", exist_ok=True)  # make a new folder to hold our scraped website HTML data. exist_ok = True means, if the directory already exists, do nothing and don't return an error

def fetch_page(url):  # declaring a new function, which takes a URL to be scraped, and a destination filename, as arguments
    global scraped_pages_count  # use the scraped_pages_count from the top of the script, instead of making a new local one
    safe_name = url.replace("https://", "").replace("/", "_")  # turn the url into a safe filename
    filename = f"{SCRAPED_PAGES_DIRECTORY}{safe_name}.html"  # name the saved file after its url, not a counter
    if os.path.exists(filename):  # check to see if file already exists, if so, don't download it from the web, load it instead
        scraped_pages_count = scraped_pages_count + 1
        with open(filename, "r", encoding="utf-8") as file:
            return file.read()  # in this case, return provides the read file data, and escapes from the function, ignoring everything below

    response = requests.get(url, headers=headers) # grab the file, using our pre-defined headers as the header data

    if not response.ok: # if the response reports an error
        print(url, "failed with status code", response.status_code) # tell us why it failed
        return None # return a "None" object, since there's nothing usable in the response, and escape from this function
    with open(filename, "w",  encoding="utf-8") as file:
        file.write(response.text)
        scraped_pages_count = scraped_pages_count + 1
    sleep(10) # wait 10 seconds, since our robots.txt file asks for a 10 second Crawl-delay
    return response.text

## Now we need to read one website and see what we can extract from it

In [15]:
# Read data from our URL and interpret it as a "soup".

#specify the URL we want to scrap
url="https://mapping-museums.bbk.ac.uk/2026/09/29/ten-years-of-mapping-museums/"

# use that URL to retrieve data from the Internet
html= fetch_page(url)

# take the retrieved data and convert it into a usable "soup" object.
soup = BeautifulSoup(html, "html.parser")

print(len(html)) #confirms we have confirmed html data by showing us how many characters are in the data

156037


I know that the blog post contents are in a div with the class "entry-content". I also know that the header is in an element with a class called "entry-title".

In [16]:
title_element = soup.find(class_="entry-title") # this is the element containing the title

entry_title = title_element.get_text()

entry_content_element = soup.find(class_="entry-content") # element containing all the paragraphs that make up the blog post

children = entry_content_element.find_all(True, recursive = False) # grab all the child elements but not the grandchildren

entry_paragraphs = [] # an empty list to hold onto all the paragraphs in the blog post

for child in children:
    entry_paragraphs.append(child.get_text())

entry_dictionary = {
    "title": entry_title,
    "paragraphs": entry_paragraphs
}

print(entry_dictionary)

{'title': 'Ten years of Mapping Museums', 'paragraphs': ['It is ten years ago this month that the first Mapping Museums project started. In September 2016, Alexandra Poulovassilis, Jamie Larkin, and I began the process of collecting, modelling, and analysing data on the UK museums sector since 1960. Andrea Ballatore soon joined, bringing expertise in geography and data analysis. Since then, the team has grown, and we’ve completed three major projects. I want to take a moment to think about what we’ve achieved and what is left to do.', 'The Mapping Museums database showed that the UK museum sector was bigger and more varied than assumed, and it made non-national museums, and especially local authority and independent museums more visible within the sector than had previously been the case. It was also the first time that non-accredited museums had been documented. When the database first launched in 2020, Lisa Ollerhead (Senior policy advisor for museums at the Department of Digital, Cu

I am now creating a function that will allow me to repeat the above process for multiple websites. A function is a way of bundling together processes that will be repeated.

In [17]:
def scrape_data(html, url):
    soup = BeautifulSoup(html, "html.parser") # convert the html in the function into a "soup"
    title_element = soup.find(class_="entry-title") # this is the element containing the title

    entry_title = title_element.get_text() #creating a variable containing a text of the title
    
    entry_content_element = soup.find(class_="entry-content") # element containing all the paragraphs that make up the blog post
    
    children = entry_content_element.find_all(True, recursive = False) # grab all the child elements but not the grandchildren
    
    entry_paragraphs = [] # an empty list to hold onto all the paragraphs in the blog post
    
    for child in children: # for every paragraph in the entry content...
        entry_paragraphs.append(child.get_text()) # append the paragraph to the list of entry paragraphs
    
    entry_dictionary = {
        "title": entry_title,
        "paragraphs": entry_paragraphs,
        "url": url
    }
    
    return(entry_dictionary)

In [18]:
#specify the URL we want to scrap
url="https://mapping-museums.bbk.ac.uk/2026/09/22/archiving-websites-about-closed-museums/"

# use that URL to retrieve data from the Internet
html= fetch_page(url)

# print(scrape_data(html, url))

In the next step we will make a list of urls and then use a for loop to go through each url and build a dictionary from the scraped urls

In [19]:
urls = [
    "https://mapping-museums.bbk.ac.uk/2026/09/29/ten-years-of-mapping-museums/",
    "https://mapping-museums.bbk.ac.uk/2026/09/22/archiving-websites-about-closed-museums/",
    "https://mapping-museums.bbk.ac.uk/2026/04/21/museums-and-the-promise-of-permanence/",
    "https://mapping-museums.bbk.ac.uk/2026/03/03/major-changes-to-the-mapping-museums-lab-database/",
    "https://mapping-museums.bbk.ac.uk/2026/01/28/new-article-on-museum-closure-data/"
]

scraped_data = [] # this is an empty list to hang onto all the dictionaries of the scraped websites

for url in urls: # for each of the urls in my list of urls...
    html= fetch_page(url) # fetch the page, retrieving it from the web and loading it as an html string...
    print("fetched", url)
    website_data = scrape_data(html, url) # scrape the data in that html string using scrape_data
    scraped_data.append(website_data) # append that scraped data to the scraped data list

print(scraped_data)

fetched https://mapping-museums.bbk.ac.uk/2026/09/29/ten-years-of-mapping-museums/
fetched https://mapping-museums.bbk.ac.uk/2026/09/22/archiving-websites-about-closed-museums/
fetched https://mapping-museums.bbk.ac.uk/2026/04/21/museums-and-the-promise-of-permanence/
fetched https://mapping-museums.bbk.ac.uk/2026/03/03/major-changes-to-the-mapping-museums-lab-database/
fetched https://mapping-museums.bbk.ac.uk/2026/01/28/new-article-on-museum-closure-data/
[{'title': 'Ten years of Mapping Museums', 'paragraphs': ['It is ten years ago this month that the first Mapping Museums project started. In September 2016, Alexandra Poulovassilis, Jamie Larkin, and I began the process of collecting, modelling, and analysing data on the UK museums sector since 1960. Andrea Ballatore soon joined, bringing expertise in geography and data analysis. Since then, the team has grown, and we’ve completed three major projects. I want to take a moment to think about what we’ve achieved and what is left to do

In [20]:
print(len(scraped_data))

5


To save a list of dictionaries as a JSON file that we can load in other contexts we use the json.dump function

In [21]:
with open("scraped_data.json", "w", encoding = "utf-8") as file: # create a new file, refer to it as file
    json.dump(scraped_data, file, indent=4, ensure_ascii=False)
    

# My Own Exploration: Scraping The Marginalian website, authored by Maria Popova 

Background: Maria Popova has been publishing a newsletter (first called Brain Pickings, then The Marginalian) since 2006. She appears to be a one-woman show, using the platform to explore wide-ranging themes touching on literature, science, philosophy, art, and all manner of things. In her own words, The Marginalian is "animated by the ultimate question that binds us all: What is all this?"

In this query, I am scraping the website for all the entries that Popova explicitly tagged with the phrase "Fyodor Dostoyevsky". She has written about, or referred to him, often over the years, so I'm curious to see how many posts she's written referencing Dostoyevsky, and what they contain. When I did a search for all entries that contained the word "Dostoyevsky", I got about 11 pages (assuming about 10 entries per page, that's 110 entries).

## Read data from one Dostoyevsky-related URL and interpret it as a "soup".

In [22]:
#specify the URL we want to scrape
url = "https://www.themarginalian.org/2023/02/25/dostoyevsky-animals-love/"

# use that URL to retrieve data from the Internet
html = fetch_page(url)

# take the retrieved data and convert it into a usable "soup" object.
soup = BeautifulSoup(html, "html.parser")

print(len(html)) # confirms we have html data by showing how many characters are in it

73352


## Inspect url and map the structure

I've noted the following:
- The title (in this case, "Dostoyevsky on Animal Rights and the Deepest Meaning of Human Love") is marked as "entry-title"
- All the paragraphs are children of "entry_content" (note underscore, not hyphen like entry-title)

In [23]:
title_element = soup.find(class_="entry-title") # this is the element containing the title

entry_title = title_element.get_text()

entry_content_element = soup.find(class_="entry_content") # element containing all the paragraphs that make up the blog post

children = entry_content_element.find_all(True, recursive = False) # grab all the child elements but not the grandchildren

entry_paragraphs = [] # an empty list to hold onto all the paragraphs in the blog post

for child in children:
    entry_paragraphs.append(child.get_text())

entry_dictionary = {
    "title": entry_title,
    "paragraphs": entry_paragraphs
}

print(entry_dictionary)

{'title': 'Dostoyevsky on Animal Rights and the Deepest Meaning of Human Love', 'paragraphs': ['', '“Love the earth and sun and the animals,” Walt Whitman wrote in his timeless advice on living a vibrant and rewarding life — advice anchored, like his poetry, in that all-enveloping totality of goodwill that makes life worth living, advice at the heart of which is the act of unselfing; poetry largely inspired by the prose of Emerson, who had written of the “secret sympathy which connects men to all the animals, and to all the inanimate world around him.”', 'A quarter century after Leaves of Grass, Fyodor Dostoyevsky (November 11, 1821–February 9, 1881) took up this bright urgency in his final novel, The Brothers Karamazov (public library | public domain) — one of the great moral masterworks in the history of literature. ', 'Portrait of Fyodor Dostoyevsky by Vasily Perov, 1871', 'Dostoyevsky — who felt deeply the throes of personal love — contours the largest meaning of love:', 'Love ever

## Find all the relevant urls (with help from Claude)

In [24]:
# This captures the urls from the 2 pages pulled up when i searched for all entries tagged with "Fyodor Dostoyevsky"
tag_pages = [
    "https://www.themarginalian.org/tag/fyodor-dostoyevsky/",
    "https://www.themarginalian.org/tag/fyodor-dostoyevsky/page/2/"
]

urls = []  # empty list to hold the article urls

for tag_page in tag_pages: # goes through those 2 pages
    html = fetch_page(tag_page) #stores html
    soup = BeautifulSoup(html, "html.parser") # uses Beautiful Soup to structure it
    main_body = soup.find(id="main_body")  # only look in the main column, not the sidebar
    for heading in main_body.find_all("h5"):  # each article title is an h5 with a link inside
        link = heading.find("a")
        urls.append(link["href"])

print(len(urls))
print(urls)

12
['https://www.themarginalian.org/2026/10/24/dostoyevsky-execution-life/', 'https://www.themarginalian.org/2024/09/01/dostyevsky-general-letter/', 'https://www.themarginalian.org/2022/10/22/dostoyevsky-in-love/', 'https://www.themarginalian.org/2016/04/26/dostoyevsky-poverty-ambition-success-art/', 'https://www.themarginalian.org/2016/02/15/anna-dostoyevsky-reminiscences-marriage/', 'https://www.themarginalian.org/2026/10/04/dostoyevsy-good-person/', 'https://www.themarginalian.org/2023/02/25/dostoyevsky-animals-love/', 'https://www.themarginalian.org/2018/08/09/wislawa-symborska-great-love/', 'https://www.themarginalian.org/2016/03/31/dostoyevsky-reason-emotion/', 'https://www.themarginalian.org/2015/06/08/dostoyevsky-george-sand/', 'https://www.themarginalian.org/2014/12/02/dostoyevsky-good-fellows/', 'https://www.themarginalian.org/2014/11/11/dostoyevsky-dream/']


In [25]:
def scrape_data(html, url):
    soup = BeautifulSoup(html, "html.parser")  # turn the html into a searchable soup

    title_element = soup.find(class_="entry-title")  # the title (hyphen)
    entry_content_element = soup.find(class_="entry_content")  # the article body (underscore!)

    if title_element is None or entry_content_element is None:  # if either is missing...
        print("Couldn't find title or content on", url)  # ...say so...
        return None  # ...and skip this page

    entry_title = title_element.get_text().strip()  # .strip() trims spaces only at the start and end

    children = entry_content_element.find_all(True, recursive=False)  # the direct children (paragraphs, quotes, etc.)

    entry_paragraphs = []
    for child in children:
        text = child.get_text().strip()
        if text != "":  # skip empty ones, like the paragraph that only holds the book cover image
            entry_paragraphs.append(text)

    entry_dictionary = {
        "title": entry_title,
        "paragraphs": entry_paragraphs,
        "url": url
    }

    return entry_dictionary

## Get the header of each entry and the number of paragraphs

In [27]:
scraped_data = []

for url in urls:
    html = fetch_page(url)
    if html is None:  # fetch_page returns None if the download failed
        continue  # skip to the next URL
    print("fetched", url) # In real time, print the url of each entry as it's found
    article = scrape_data(html, url)
    if article is not None:  # only keep pages that scraped successfully
        scraped_data.append(article)

print(len(scraped_data), "articles scraped") # print how many entries were found tagged with "Fyodor Dostoyevsky"

with open("dostoyevsky_articles.json", "w", encoding="utf-8") as file:
    json.dump(scraped_data, file, indent=4, ensure_ascii=False)

for article in scraped_data:
    print(article["title"], "-", len(article["paragraphs"]), "paragraphs") # Print the entry title with the # of paragraphs contained

fetched https://www.themarginalian.org/2026/10/24/dostoyevsky-execution-life/
fetched https://www.themarginalian.org/2024/09/01/dostyevsky-general-letter/
fetched https://www.themarginalian.org/2022/10/22/dostoyevsky-in-love/
fetched https://www.themarginalian.org/2016/04/26/dostoyevsky-poverty-ambition-success-art/
fetched https://www.themarginalian.org/2016/02/15/anna-dostoyevsky-reminiscences-marriage/
fetched https://www.themarginalian.org/2026/10/04/dostoyevsy-good-person/
fetched https://www.themarginalian.org/2023/02/25/dostoyevsky-animals-love/
fetched https://www.themarginalian.org/2018/08/09/wislawa-symborska-great-love/
fetched https://www.themarginalian.org/2016/03/31/dostoyevsky-reason-emotion/
fetched https://www.themarginalian.org/2015/06/08/dostoyevsky-george-sand/
fetched https://www.themarginalian.org/2014/12/02/dostoyevsky-good-fellows/
fetched https://www.themarginalian.org/2014/11/11/dostoyevsky-dream/
12 articles scraped
Dostoyevsky, Just After His Death Sentence 